# 03 — Baseline CNN

**Workstream:** Disease Detection · **Batch 5 (Day 5)** · **Status: IMPLEMENTED**

Simple CNN trained from scratch (`ai/disease_detection/src/baseline_cnn.py`) — the floor that transfer
learning (notebook 04) has to beat.

**Sandbox note:** this run uses `plantvillage_fast_subset/` (60 real images/class = 780 total, a random
draw from the full 220/class real subset) and few epochs, purely to fit CPU training time inside this
session. The full 220/class subset (and the actual per-class counts up to 5,357 — see
`docs/disease/class-shortlist.md`) is already on disk at `plantvillage_subset/` for a proper run later on
better hardware — swap the `ROOT` path below and increase `epochs` to use it.

In [1]:
import sys, time
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ai" / "common" / "config.py").exists())
sys.path.insert(0, str(_root))
import torch
from torch.utils.data import DataLoader
from ai.disease_detection.src.dataset import index_dataset, stratified_split, PlantVillageSubset
from ai.disease_detection.src.transforms import train_transform, eval_transform
from ai.disease_detection.src.baseline_cnn import build_baseline_cnn
from ai.disease_detection.src.train import fit

ROOT = _root / "data/raw/disease_images/plantvillage_fast_subset"
paths, labels, class_to_idx = index_dataset(ROOT)
print(f"{len(paths)} images, {len(class_to_idx)} classes")

train_idx, val_idx, test_idx = stratified_split(paths, labels, seed=42)
def subset(idx_list, transform):
    return PlantVillageSubset([paths[i] for i in idx_list], [labels[i] for i in idx_list], class_to_idx, transform)
train_ds, val_ds, test_ds = subset(train_idx, train_transform), subset(val_idx, eval_transform), subset(test_idx, eval_transform)
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=1)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=1)
device = torch.device("cpu")

780 images, 13 classes
train=546  val=117  test=117


## Train

In [2]:
model = build_baseline_cnn(num_classes=len(class_to_idx)).to(device)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

t0 = time.time()
history = fit(model, train_loader, val_loader, device, epochs=4, lr=1e-3)
print(f"\nTotal training time: {time.time()-t0:.0f}s")

Trainable params: 897869


epoch 1/4  train_loss=2.5321  val_loss=2.2549  val_acc=0.2051  (50.6s)


epoch 2/4  train_loss=2.1885  val_loss=1.7496  val_acc=0.3761  (50.0s)


epoch 3/4  train_loss=1.9341  val_loss=1.7186  val_acc=0.3590  (50.1s)


epoch 4/4  train_loss=1.7692  val_loss=1.4966  val_acc=0.4615  (50.0s)

Total training time: 201s


## Save

In [3]:
import pickle
MODEL_DIR = _root / "ai" / "disease_detection" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
with open(MODEL_DIR / "baseline_cnn_REAL_fastsubset.pkl", "wb") as f:
    pickle.dump({"state_dict": model.state_dict(), "class_to_idx": class_to_idx, "history": history}, f)
print("Saved baseline CNN + class mapping + training history.")
best_val_acc = max(h["val_acc"] for h in history)
print(f"Best val_acc reached: {best_val_acc:.3f}")

Saved baseline CNN + class mapping + training history.
Best val_acc reached: 0.462


## Next: `04_transfer_learning_training.ipynb`

**Status: IMPLEMENTED, real training ran** (not simulated) — see actual epoch-by-epoch numbers above, on
real (not synthetic) images. `PROGRESS_LEDGER.md` records the exact val_acc reached.